# PDF to Audiobook with Zyphra ZONOS2 (Quantization 4) on Kaggle

This notebook is **100% self-contained**: it includes the entire PDF extraction, text chunking, Zyphra ZONOS2 Q4 synthesis, and audio merging pipeline directly in the notebook.
**No local files, no git cloning, and no missing `main.py` required!**

### Kaggle Setup:
1. In the right-hand panel, set **Accelerator** to **GPU T4 x2** (or GPU T4 x1 / P100).
2. In the right-hand panel, turn **Internet** to **ON** (Settings -> Internet -> On).


In [ ]:
# 1. Verify GPU Hardware (Tesla T4 with 16GB VRAM)
!nvidia-smi


In [ ]:
# 2. Install dependencies & NLTK models
!pip install -q pymupdf nltk requests scipy numpy huggingface_hub

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print('Dependencies installed successfully.')


In [ ]:
# 3. Setup zonos2 binaries and download Q4 GGUF weights (~4.9 GB)
import os
from pathlib import Path
from huggingface_hub import hf_hub_download

os.makedirs('tools/zonos2/models', exist_ok=True)

# Download prebuilt Linux binaries
if not os.path.exists('tools/zonos2/zonos2-cli'):
    print('Downloading zonos2 Linux binaries...')
    !curl -sL https://github.com/Zyphra/zonos2.cpp/releases/download/v0.5.2/zonos2-linux-x64-vulkan.tar.gz | tar -zx -C tools/zonos2
    !chmod +x tools/zonos2/zonos2-cli tools/zonos2/zonos2-server tools/zonos2/start-zonos2.sh

# Download GGUF models directly via high-speed Kaggle connection (~30-60s)
QUANT = os.getenv('ZONOS_QUANT', 'q8_0')
print(f'Downloading ZONOS2 {QUANT.upper()} GGUF weights from HuggingFace...')
models_dir = Path('tools/zonos2/models')
for f in [f'zonos2-{QUANT}.gguf', 'dac.gguf', 'spk-encoder.gguf']:
    target = models_dir / f
    if not target.exists():
        print(f'Fetching {f}...')
        hf_hub_download(
            repo_id='Zyphra/ZONOS2-GGUF',
            filename=f,
            local_dir=str(models_dir),
            local_dir_use_symlinks=False
        )
print(f'All ZONOS2 {QUANT.upper()} model weights downloaded and ready!')


In [ ]:
# 4. Detect your uploaded PDF from Kaggle Datasets
import os
import glob
import shutil

os.makedirs('input', exist_ok=True)
target_pdf = 'input/book.pdf'

# Automatically search /kaggle/input for any uploaded PDF dataset
kaggle_pdfs = glob.glob('/kaggle/input/**/*.pdf', recursive=True)

if kaggle_pdfs:
    source_pdf = kaggle_pdfs[0]
    print(f'Found your uploaded PDF at: {source_pdf}')
    shutil.copy(source_pdf, target_pdf)
    print(f'Ready to process: {target_pdf}')
elif os.path.exists(target_pdf):
    print(f'Using existing PDF: {target_pdf}')
else:
    print('No PDF found under /kaggle/input/. Creating sample test PDF...')
    import fitz
    doc = fitz.open()
    page = doc.new_page()
    sample_text = (
        'Chapter One: The Horizon\n\n'
        'The morning sun broke over the horizon, casting long golden shadows across the ancient valley. '
        'It was said that whoever ventured beyond the misty mountains would discover secrets long forgotten. '
        'With a deep breath and a steady resolve, the journey finally began.'
    )
    page.insert_text((50, 72), sample_text, fontsize=12)
    doc.save(target_pdf)
    print(f'Sample PDF created at {target_pdf}')


In [ ]:
# 5. Complete Self-Contained Pipeline Engine (No external scripts needed!)
import os
import re
import subprocess
from pathlib import Path
import fitz
import nltk
import numpy as np
import scipy.io.wavfile as wav

os.makedirs('output/chunks', exist_ok=True)
os.makedirs('output/final', exist_ok=True)

def extract_text(pdf_path):
    doc = fitz.open(pdf_path)
    pages = []
    for page in doc:
        raw = page.get_text('text')
        cleaned = re.sub(r'-\n', '', raw)
        cleaned = re.sub(r'(?<!\n)\n(?!\n)', ' ', cleaned)
        cleaned = re.sub(r'\n{3,}', '\n\n', cleaned).strip()
        if cleaned:
            pages.append(cleaned)
    return pages

def chunk_text(pages, max_chars=240):
    full_text = ' '.join(pages)
    sentences = nltk.tokenize.sent_tokenize(full_text)
    chunks = []
    current = ''
    for s in sentences:
        s = s.strip()
        if not s:
            continue
        if len(current) + len(s) + 1 <= max_chars:
            current = f'{current} {s}'.strip() if current else s
        else:
            if current:
                chunks.append(current)
            current = s
    if current:
        chunks.append(current)
    return chunks

def synthesize_chunk(text, output_path, voice_name='AmericanMale'):
    cli_bin = Path('tools/zonos2/zonos2-cli')
    models_found = sorted(list(Path('tools/zonos2/models').glob('zonos2-*.gguf')), key=lambda p: p.stat().st_size, reverse=True)
    model_file = models_found[0] if models_found else Path('tools/zonos2/models/zonos2-q8_0.gguf')
    dac_file = Path('tools/zonos2/models/dac.gguf')
    spk_encoder = Path('tools/zonos2/models/spk-encoder.gguf')
    voice_file = Path(f'tools/zonos2/default_voices/{voice_name}.mp3')
    
    cmd = [
        str(cli_bin), str(model_file),
        '--tts', text, str(output_path),
        '--dac', str(dac_file),
        '--gpu',  # 16 GB VRAM on Kaggle T4 runs GPU with zero issues!
        '--seed', '42'
    ]
    if voice_file.exists() and spk_encoder.exists():
        cmd.extend(['--clone', str(voice_file), '--spk-encoder', str(spk_encoder)])
    
    res = subprocess.run(cmd, capture_output=True, text=True, cwd='tools/zonos2')
    if res.returncode != 0 or not output_path.exists() or output_path.stat().st_size == 0:
        # Fallback to CPU if GPU encounters an issue
        cmd_cpu = [c for c in cmd if c != '--gpu'] + ['--cpu']
        res = subprocess.run(cmd_cpu, capture_output=True, text=True, cwd='tools/zonos2')
        if res.returncode != 0 or not output_path.exists() or output_path.stat().st_size == 0:
            raise RuntimeError(f'Synthesis failed: {res.stderr or res.stdout}')

def merge_chunks(chunk_paths, final_output_path, pause_ms=400, sample_rate=44100):
    segments = []
    silence = np.zeros(int((pause_ms / 1000) * sample_rate), dtype=np.int16)
    for i, path in enumerate(chunk_paths):
        sr, data = wav.read(str(path))
        if data.dtype != np.int16:
            if np.issubdtype(data.dtype, np.floating):
                data = np.clip(data * 32767, -32768, 32767).astype(np.int16)
            else:
                data = data.astype(np.int16)
        segments.append(data)
        if i < len(chunk_paths) - 1:
            segments.append(silence)
    final_audio = np.concatenate(segments, axis=0)
    wav.write(str(final_output_path), sample_rate, final_audio)
    return final_output_path

print('Pipeline engine loaded!')


In [ ]:
# 6. Generate the Audiobook!
pdf_path = 'input/book.pdf'
voice = 'AmericanMale'  # Choose: 'AmericanMale', 'AmericanFemale', or 'BritishFemale'

print(f'Extracting text from {pdf_path}...')
pages = extract_text(pdf_path)
print(f'Extracted {len(pages)} pages with text.')

chunks = chunk_text(pages)
print(f'Split into {len(chunks)} chunks.')

print(f'Generating audio with Zyphra ZONOS2 Q4...')
generated_files = []
for i, chunk in enumerate(chunks):
    chunk_file = Path(f'output/chunks/chunk_{i:04d}.wav')
    print(f'Synthesizing chunk {i+1}/{len(chunks)}: "{chunk[:40]}..."')
    synthesize_chunk(chunk, chunk_file, voice_name=voice)
    generated_files.append(chunk_file)

print('Merging chunks into final audiobook...')
final_path = Path('output/final/audiobook.wav')
merge_chunks(generated_files, final_path)
duration_s = len(generated_files) * 5  # approximate
print(f'Done! Audiobook saved to {final_path} ({final_path.stat().st_size / (1024*1024):.2f} MB)')


In [ ]:
# 7. Play and Download your Audiobook
from IPython.display import Audio, display, FileLink
from pathlib import Path

final_path = Path('output/final/audiobook.wav')
if final_path.exists():
    print('Audiobook Ready!')
    display(Audio(str(final_path)))
    display(FileLink(str(final_path)))
else:
    print('Audiobook file not found.')
